# Bias Analysis

Testing whether the differences found in EDA are statistically significant or just noise.
* Early access reviewers vs regular reviewers
* Free copy recipients vs purchasers
* Playtime vs recommendation (Mann-Whitney + threshold analysis)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_parquet('../data/steam_reviews_sampled.parquet')

### Early Access Bias
* Early access reviewers recommend at 82% vs 88% for regular reviewers
* Chi-squared statistic: 6530.45, p-value: 0.0. Statistically significant
* Cramér's V: 0.050. Weak effect size
* The bias is real but small. Early access alone doesn't strongly determine recommendation.

In [2]:
pd.crosstab(df['written_during_early_access'], df['recommended'])

recommended,False,True
written_during_early_access,,
False,283891,2083655
True,42851,199280


In [3]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['written_during_early_access'],df['recommended'])
stat,p,_,_ = chi2_contingency(table)

print(f'Chi-squared statistics: {stat:.2f}')
print(f'P-value: {p}')

Chi-squared statistics: 6530.45
P-value: 0.0


In [4]:
n = table.sum().sum()
cramers_v = np.sqrt(stat/(n*(min(table.shape)-1)))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.0500


### Free Copy Bias
* Free copy recipients recommend at 89% vs 87% for purchasers
* Chi-squared statistic: 155.14, p-value: ~0.0. Statistically significant
* Cramér's V: 0.008. Negligible effect size
* Difference exists statistically but is practically meaningless

In [5]:
pd.crosstab(df['received_for_free'],df['recommended'])

recommended,False,True
received_for_free,,
False,317604,2209790
True,9138,73145


In [6]:
table = pd.crosstab(df['received_for_free'],df['recommended'])
stat,p,_,_ = chi2_contingency(table)

print(f'Chi-squared statistics: {stat:.4f}')
print(f'P-value: {p}')

Chi-squared statistics: 155.1427
P-value: 1.3031647424953305e-35


In [7]:
n = table.sum().sum()
cramers_v = np.sqrt(stat/(n*(min(table.shape)-1)))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.0077


### Playtime vs Recommendation
* Negative reviewers: median 37h, positive reviewers: median 31h
* Mann-Whitney U test: p-value ~0.0. Statistically significant
* Rank-biserial correlation: 0.026. Weak effect size
* Playtime difference is real but small as a standalone predictor

In [8]:
from scipy.stats import mannwhitneyu

df['playtime_hours'] = df['author.playtime_at_review'] / 60

pos = df[df['recommended'] == True]['playtime_hours'].dropna()
neg = df[df['recommended'] == False]['playtime_hours'].dropna()

stat,p = mannwhitneyu(pos,neg)
print(f'U statistic: {stat:.2f}')
print(f'P-value: {p}')

U statistic: 362350216093.50
P-value: 1.599971649794195e-125


In [9]:
n1,n2 = len(pos),len(neg)
r = 1 - (2 * stat) / (n1 * n2)
print(f'Rank-biserial correlation: {r:.4f}')

Rank-biserial correlation: 0.0258


### Playtime Sentiment Threshold
* 0-5 hours: 82% recommend. Quick quitters, didn't engage enough
* 5-100 hours: ~90-91%. Sweet spot, highest satisfaction
* 100-500 hours: 85%. Declining satisfaction
* 500+ hours: 80%. Most invested players are least satisfied
* Non-linear (inverted U-shape) relationship. Supports using tree-based models over linear

In [10]:
df['playtime_bin'] = pd.cut(df['playtime_hours'], bins = [0,5,10,25,50,100,500,float('inf')])
df.groupby('playtime_bin')['recommended'].mean()

playtime_bin
(0.0, 5.0]       0.82
(5.0, 10.0]      0.91
(10.0, 25.0]     0.91
(25.0, 50.0]     0.91
(50.0, 100.0]    0.90
(100.0, 500.0]   0.85
(500.0, inf]     0.80
Name: recommended, dtype: float64

## Summary
* All three biases (early access, free copy, playtime) are statistically significant, not noise
* But all have weak effect sizes (Cramér's V < 0.05, rank-biserial ~0.03)
* No single behavioral feature strongly predicts recommendation on its own
* Playtime-sentiment relationship is non-linear (inverted U-shape). Strongest satisfaction between 5-100 hours
* Combined features and non-linear models will be needed for prediction